In [2]:
import faiss
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR    = Path("../data/processed")
MATRIX_PATH = DATA_DIR / "feature_matrix.parquet"
CLEAN_PATH  = DATA_DIR / "filings_clean.parquet"

fm = pd.read_parquet(MATRIX_PATH)

print(f"Feature matrix: {fm.shape[0]} orgs, {fm.shape[1]} columns")
print(f"Columns: {fm.columns.tolist()}")
fm.head(3)

Feature matrix: 344 orgs, 11 columns
Columns: ['ein', 'org_name', 'org_state', 'org_ntee_code', 'tax_prd_yr', 'program_expense_ratio', 'admin_overhead_ratio', 'net_asset_margin', 'labor_cost_ratio', 'solvency_ratio', 'revenue_yoy_growth']


,ein,org_name,org_state,org_ntee_code,tax_prd_yr,program_expense_ratio,admin_overhead_ratio,net_asset_margin,labor_cost_ratio,solvency_ratio,revenue_yoy_growth
0,042632526,Elderhostel Inc,MA,N500,2024,1.013286,-0.932392,-0.542861,-0.951506,-0.537092,-0.269264
1,042775991,Metropolitan Boston Housing Partnership Inc,MA,L20,2024,0.866517,-0.800522,-0.672369,-1.268384,-0.553633,0.430018
2,341747398,American Endowment Foundation,OH,T31,2023,-1.457823,1.398050,2.181113,-1.426526,2.923457,-0.041517


- FAISS only accepts a raw float32 numpy matrix — no strings, no metadata.
- We isolate the numeric feature columns and keep a separate lookup table to map FAISS result indices back to EINs and org names.

In [ ]:
# Metadata columns 
META_COLS = ["ein", "org_name", "org_state", "org_ntee_code", "tax_prd_yr"]

# Everything else is a feature
FEATURE_COLS = [c for c in fm.columns if c not in META_COLS]

print(f"Feature columns ({len(FEATURE_COLS)}):")
for col in FEATURE_COLS:
    print(f"  {col}")

# Lookup table — FAISS returns integer indices, we map them back to org info
lookup = fm[META_COLS].reset_index(drop=True)

# Float32 matrix — FAISS requirement
vectors = fm[FEATURE_COLS].values.astype(np.float32)

print(f"\nVector matrix shape: {vectors.shape}")
print(f"dtype: {vectors.dtype}")

Feature columns (6):
  program_expense_ratio
  admin_overhead_ratio
  net_asset_margin
  labor_cost_ratio
  solvency_ratio
  revenue_yoy_growth

Vector matrix shape: (344, 6)
dtype: float32


`IndexFlatL2` is the simplest FAISS index; it computes exact L2 (Euclidean)
distance between every query vector and every vector in the index.

For 344 orgs this is instant. At 100k+ orgs you'd swap to `IndexIVFFlat`
for approximate search with much faster query times.

In [ ]:
n_features = vectors.shape[1]

# Build index
index = faiss.IndexFlatL2(n_features)

# Add all org vectors to the index
index.add(vectors)

print(f"FAISS index built")
print(f"  Index type:    IndexFlatL2")
print(f"  Dimensions:    {n_features}")
print(f"  Vectors stored: {index.ntotal}")

Takes an EIN, finds its vector in the matrix, queries FAISS for the
top K nearest neighbors, and returns a clean DataFrame of results.

The query org itself is always the closest match (distance=0) so we
skip index 0 and return K+1 results to get K true lookalikes.

In [ ]:
def find_lookalikes(ein: str, k: int = 5) -> pd.DataFrame:
    """
    Find the top-K most financially similar orgs to a given EIN.

    Parameters
    ----------
    ein : str   Nine-digit EIN string (zero-padded)
    k   : int   Number of lookalikes to return (default 5)

    Returns
    -------
    pd.DataFrame with columns:
        rank, ein, org_name, org_state, org_ntee_code, tax_prd_yr, l2_distance
    """
    # Find the row index for this EIN
    matches = lookup[lookup["ein"] == ein]
    if matches.empty:
        raise ValueError(f"EIN {ein} not found in feature matrix")

    idx = matches.index[0]

    # Extract the query vector — shape must be (1, n_features) for FAISS
    query_vector = vectors[idx].reshape(1, -1)

    # Search — k+1 because the query org itself is always result #0
    distances, indices = index.search(query_vector, k + 1)

    # Skip the first result (the query org itself, distance=0)
    result_indices   = indices[0][1:]
    result_distances = distances[0][1:]

    # Build results DataFrame
    results = lookup.iloc[result_indices].copy()
    results["l2_distance"] = result_distances
    results["rank"] = range(1, k + 1)

    return results[["rank", "ein", "org_name", "org_state", "org_ntee_code", "tax_prd_yr", "l2_distance"]].reset_index(drop=True)


print("find_lookalikes() ready")

Testing: find top 5 lookalikes for the first org

In [ ]:
# Use the first org in the dataset as a test query
test_ein  = lookup.iloc[0]["ein"]
test_name = lookup.iloc[0]["org_name"]

print(f"Query org: {test_name} (EIN: {test_ein})")
print()

results = find_lookalikes(test_ein, k=5)
results

Pull the raw feature vectors for the query org and its top 5 lookalikes
to understand which financial ratios drove the similarity.

In [ ]:
# Collect EINs: query org + its 5 lookalikes
all_eins = [test_ein] + results["ein"].tolist()

# Pull their feature vectors from the matrix
comparison = fm[fm["ein"].isin(all_eins)][["ein", "org_name"] + FEATURE_COLS].copy()

# Tag the query org
comparison["role"] = comparison["ein"].apply(
    lambda x: "QUERY" if x == test_ein else "lookalike"
)

comparison = comparison.set_index(["role", "org_name"])
comparison